In [ ]:
import numpy as np
from scipy.io import loadmat
import mne

# change this to wherever you have your data stored
baseDir = "/Users/apple/Documents/Neurotech/Cursor/Replicating Chavarriaga"

In [ ]:
'''
    the first thing to do is load the data.
    this data was meant to be opened with matlab. fortunately, the 'loadmat' function from scipy allows us to open such files!
'''

# change this to whichever subject and session's data you have downloaded
chavDat = loadmat(baseDir + "/Subject02_s1.mat")

Your next steps should be to format the data and create an MNE Epochs object (https://mne.tools/stable/generated/mne.Epochs.html#mne.Epochs.equalize_event_counts) out of it. This will make the rest of our job a lot easier, but making the epochs object itself can be a frustrating process.

You should start by understanding the shape of the data. 'chavDat' (you can change this name) has relevant data under 'run'. Once you extract this data, you'll see that it still looks confusing. 

Try running the cell below. The [0] is an unfortunate consequence of trying to use a Matlab data file in Python; sometimes the data will be formatted inefficiently. You'll have to figure out how to adapt to these.

In [152]:
rawEEG = np.concatenate(chavDat['run'][0], axis=0) # shape: (91648, 64)

Notice how rawEEG has 10 rows. Try to figure out what each row corresponds to (hint: re-read the procedure!)

(Read this after attempting the above)

You'll notice that, with each participant, in each session, the researchers did ten blocks of the experiment. As such, rawEEG[0] corresponds to the first block, and so on. For now, we'll analyze each block individually, and then combine them together.

Your next step should be to analyze each block and separate EEG data from event labels. You'll have to form a MNE Raw object, see this website for documentation: https://mne.tools/stable/generated/mne.io.RawArray.html

In [119]:
# extract eeg and events
unformattedRawEEGdata = []
unformattedRawEEGevents = []

for i in range(len(rawEEG)): # 10 blocks per session
    unformattedRawEEGdata.append(rawEEG[i][0][0])
    unformattedRawEEGevents.append(rawEEG[i][0][1][0][0][4][0][0])

In [75]:
# get metadata
samplingRate = 512

channelLabels = rawEEG[0][0][1][0][0][3]
channelLabels = [label[0][0] for label in channelLabels]
channelLabels = channelLabels[:-1] # remove last channel (not used)

In [79]:
infoObjForMNE = mne.create_info(
    ch_names=channelLabels,
    sfreq=samplingRate,
    ch_types=['eeg']*64
)

In [146]:
block1rawMNE = mne.io.RawArray(np.array(unformattedRawEEGdata[0]).reshape(64, -1), infoObjForMNE)

Creating RawArray with float64 data, n_channels=64, n_times=91648
    Range : 0 ... 91647 =      0.000 ...   178.998 secs
Ready.


In [145]:
block1eventsRaw = np.concatenate(
        (
            unformattedRawEEGevents[0][0], 
            [[0]]*len(unformattedRawEEGevents[0][0]), 
            unformattedRawEEGevents[0][1]
        ), axis=1
     )

eventsToConsider = [5, 6, 9, 10] # 5 & 10 are correct trials, 6 & 9 are incorrect

In [ ]:
block1epochs = mne.Epochs(
    raw=block1rawMNE,
    events=block1eventsRaw,
    event_id=eventsToConsider,
    tmax=2.0 # trials last approx 2000 ms (2s)
)

Not setting metadata
51 matching events found
Setting baseline interval to [-0.19921875, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated


In [ ]:
# next steps: 